# Day 3-02｜Track ID 產生與穩定性檢查
> Python 籃球運動資料分析課程  
> 使用模擬的多影格 boxes 產生 track_id，說明導入 ByteTrack 前所需的追蹤資料結構。  
> 修課背景：具備基礎 Python 語法即可；不預設電腦視覺或運動資料分析經驗。

## 學習目標
- 理解 track state 包含 ID 與最新位置。
- 用簡化中心點配對維持跨 frame ID。
- 輸出可供 BEV 投影使用的 track table。

## 完成產出
- 一份包含 frame、track_id 與 bbox 的追蹤結果。

## 課堂要求
- 按照本單元順序執行各段程式。
- 僅修改題目指定的變數、路徑或參數。
- 完成指定輸出後，記錄結果並供課堂討論。


## 課程流程
1. 載入多 frame 範例 boxes。
2. 執行簡化 tracker。
3. 視覺化其中一個 frame 的 track_id。


In [ ]:
from pathlib import Path
import sys

# Colab 重新啟動 runtime 後，先掛載學生自己的 Google Drive。
try:
    from google.colab import drive  # type: ignore[import-not-found]

    IN_COLAB = True
    drive.mount("/content/drive")
except ModuleNotFoundError:
    IN_COLAB = False

COURSE_ROOT = Path("/content/drive/MyDrive/basketball_hackathon/course")
if not COURSE_ROOT.exists():
    # 本機驗證或 zip 解壓後執行時，從目前資料夾往上找課程根目錄。
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "src").exists() and (candidate / "assets").exists():
            COURSE_ROOT = candidate
            break

if not COURSE_ROOT.exists():
    raise FileNotFoundError("找不到課程資料夾，請先執行 init_colab.ipynb。")

if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

from src.course_setup import install_requirements_if_colab, print_environment_summary  # noqa: E402

install_requirements_if_colab(COURSE_ROOT)
print_environment_summary(COURSE_ROOT)


In [ ]:
import numpy as np
import pandas as pd
from src.cv_utils import load_json, read_image_rgb, draw_boxes, show_image, save_json

track_data = load_json(
    COURSE_ROOT / "assets" / "samples" / "sample_tracking_boxes.json"
)
frames = track_data["frames"]
print("frames:", len(frames))

In [ ]:
def box_center(box):
    x1, y1, x2, y2 = box
    return np.array([(x1 + x2) / 2, (y1 + y2) / 2], dtype=float)


# 簡化 tracker：第一個 frame 依序給 ID；後續用最近中心點配對。
next_id = 0
active = {}  # track_id -> center
records = []

for f in frames:
    dets = f["detections"]
    centers = [box_center(d["bbox_xyxy"]) for d in dets]
    assigned_tracks = []
    if not active:
        for c in centers:
            active[next_id] = c
            assigned_tracks.append(next_id)
            next_id += 1
    else:
        used = set()
        new_active = {}
        for c in centers:
            best_id, best_dist = None, 1e9
            for tid, old_c in active.items():
                if tid in used:
                    continue
                dist = float(np.linalg.norm(c - old_c))
                if dist < best_dist:
                    best_id, best_dist = tid, dist
            if best_id is None or best_dist > 80:
                best_id = next_id
                next_id += 1
            used.add(best_id)
            new_active[best_id] = c
            assigned_tracks.append(best_id)
        active = new_active
    for det, tid, c in zip(dets, assigned_tracks, centers):
        records.append(
            {
                "frame": f["frame_index"],
                "track_id": int(tid),
                "bbox_xyxy": det["bbox_xyxy"],
                "center_x": float(c[0]),
                "center_y": float(c[1]),
            }
        )

tracks = pd.DataFrame(records)
tracks.head()

In [ ]:
# 畫其中一個 frame 的 track_id。
image = read_image_rgb(COURSE_ROOT / "assets" / "samples" / "sample_court_frame.png")
frame_id = 8
rows = tracks[tracks["frame"] == frame_id]
boxes = rows["bbox_xyxy"].tolist()
labels = [f"ID {tid}" for tid in rows["track_id"]]
vis = draw_boxes(image, boxes, labels)
show_image(vis, f"frame {frame_id} with track IDs")

out_json = COURSE_ROOT / "assets" / "results" / "d3_02_tracks.json"
save_json(records, out_json)
print("saved:", out_json)

正式專案可改用 `supervision.ByteTrack`。本單元使用簡化版本，以檢查 track ID 的資料結構與視覺化結果。